# AutoService-LLM: QLoRA fine-tune (Step 2 of 3), fixed version
Trains `Qwen2.5-1.5B-Instruct` on the **train** split only. Validation is used for eval loss. **Test is never trained on.**

**How to run:** Runtime > **Restart session** (clears old broken state), set Runtime type to **T4 GPU**, then **Run all**. Cells must run in order. About 10-20 min.

In [ ]:
!pip -q install -U transformers accelerate peft trl bitsandbytes datasets rouge-score

In [ ]:
import torch, gc, json, os, time, random, shutil
import pandas as pd
import transformers, peft, trl

assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU, then restart."
print("GPU:", torch.cuda.get_device_name(0))
print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__, "| trl", trl.__version__)

SEED = 42
random.seed(SEED); torch.manual_seed(SEED)

In [ ]:
!rm -rf auto-service-llm
!git clone -q https://github.com/Nisha-Lohar04/auto-service-llm.git

def load(p): return [json.loads(l) for l in open(p, encoding="utf-8")]
train = load("auto-service-llm/data/processed/train.jsonl")
val   = load("auto-service-llm/data/processed/validation.jsonl")
test  = load("auto-service-llm/data/processed/test.jsonl")
print("train/val/test:", len(train), len(val), len(test), "(expected 162 / 24 / 47)")

# no group should appear in two splits
g = [{r["group_id"] for r in s} for s in (train, val, test)]
assert not (g[0] & g[1] or g[0] & g[2] or g[1] & g[2]), "group leakage between splits!"

from datasets import Dataset
def to_chat(rows):  # plain question -> answer, no custom system prompt
    return Dataset.from_list([{"messages": [{"role": "user", "content": r["question"]},
                                            {"role": "assistant", "content": r["answer"]}]} for r in rows])
train_ds, val_ds = to_chat(train), to_chat(val)
print(train_ds[0])

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)

def load_base(**kw):
    # newer transformers use `dtype`, older use `torch_dtype`
    try:
        return AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16, device_map="auto", **kw)
    except TypeError:
        return AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map="auto", **kw)

model = load_base(quantization_config=bnb)   # 4-bit weights, fp16 for everything else

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                  target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])
print("model loaded")

In [ ]:
from trl import SFTTrainer, SFTConfig

def make_cfg():
    kw = dict(output_dir="qlora-out", num_train_epochs=3, per_device_train_batch_size=4,
              gradient_accumulation_steps=2, learning_rate=2e-4, lr_scheduler_type="cosine",
              warmup_steps=3, bf16=False, fp16=True,      # T4 has no bf16
              logging_steps=5, save_strategy="no",
              report_to="none",                           # set to "wandb" later for experiment tracking
              seed=SEED)
    try:
        return SFTConfig(eval_strategy="epoch", **kw)
    except TypeError:                                     # older transformers
        return SFTConfig(evaluation_strategy="epoch", **kw)

cfg = make_cfg()
print("config ok")

In [ ]:
trainer = SFTTrainer(model=model, args=cfg, train_dataset=train_ds, eval_dataset=val_ds, peft_config=lora)

# fp16 training needs the trainable LoRA weights in fp32 (otherwise: "BFloat16 / Attempting to unscale FP16 gradients")
for p in trainer.model.parameters():
    if p.requires_grad and p.dtype != torch.float32:
        p.data = p.data.float()
dtypes = {p.dtype for p in trainer.model.parameters() if p.requires_grad}
print("trainable dtypes:", dtypes)
assert dtypes == {torch.float32}, f"unexpected trainable dtypes: {dtypes}"

trainer.train()
trainer.save_model("qlora-adapter")
print("saved adapter to qlora-adapter/")

In [ ]:
hist = pd.DataFrame(trainer.state.log_history)
cols = [c for c in ["epoch", "loss", "eval_loss"] if c in hist.columns]
print(hist[cols].dropna(how="all", subset=[c for c in cols if c != "epoch"]).to_string())
# Healthy: both losses fall. If eval_loss rises while loss keeps falling, it is overfitting.

In [ ]:
# Free the 4-bit training model, then reload the base in fp16 (same as the baseline run) and attach the adapter
for name in ["trainer", "model"]:
    if name in globals():
        del globals()[name]
gc.collect(); torch.cuda.empty_cache()

from peft import PeftModel
base = load_base()
ft = PeftModel.from_pretrained(base, "qlora-adapter")
ft.eval()
dev = next(ft.parameters()).device

@torch.no_grad()
def generate(q):
    text = tok.apply_chat_template([{"role": "user", "content": q}], tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(dev)
    t0 = time.time()
    out = ft.generate(**inputs, max_new_tokens=200, do_sample=False,
                      temperature=None, top_p=None, top_k=None, pad_token_id=tok.eos_token_id)
    dt = time.time() - t0
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip(), dt

print(generate(test[0]["question"]))

In [ ]:
from rouge_score import rouge_scorer
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

rows = []
for r in test:
    ans, dt = generate(r["question"])
    rows.append({"mode": "finetuned", "example_id": r["example_id"], "group_id": r["group_id"],
                 "origin": r["origin"], "question": r["question"], "reference": r["answer"],
                 "prediction": ans, "rougeL": scorer.score(r["answer"], ans)["rougeL"].fmeasure,
                 "latency_s": dt, "words": len(ans.split())})
dff = pd.DataFrame(rows)
summary = dff.groupby("mode")[["rougeL", "latency_s", "words"]].mean().round(3)

# compare with the saved baseline if it is in the repo
bp = "auto-service-llm/results/baseline_summary.csv"
if os.path.exists(bp):
    summary = pd.concat([pd.read_csv(bp, index_col=0), summary])
summary

In [ ]:
for _, r in dff.head(4).iterrows():
    print("Q:", r["question"]); print("REF:", r["reference"]); print("PRED:", r["prediction"]); print()

In [ ]:
os.makedirs("results", exist_ok=True)
dff.to_csv("results/finetuned_predictions.csv", index=False)
summary.to_csv("results/finetuned_summary.csv")
shutil.make_archive("qlora-adapter", "zip", "qlora-adapter")

from google.colab import files
files.download("results/finetuned_predictions.csv")
files.download("results/finetuned_summary.csv")
files.download("qlora-adapter.zip")